## **Initial setup**

Install Bambu and required packages:

In [ ]:
!echo "deb http://ppa.launchpad.net/git-core/ppa/ubuntu $(cat /etc/os-release | grep UBUNTU_CODENAME | sed 's/.*=//g') main" >> /etc/apt/sources.list.d/git-core.list
!apt-key adv --keyserver keyserver.ubuntu.com --recv-keys A1715D88E1DF1F24
!apt-get update
!apt-get install -y --no-install-recommends build-essential ca-certificates gcc-multilib g++-multilib git libtinfo5 verilator wget
!wget https://release.bambuhls.eu/appimage/bambu-latest.AppImage
!chmod +x bambu-*.AppImage
!ln -sf $PWD/bambu-*.AppImage /bin/bambu
!ln -sf $PWD/bambu-*.AppImage /bin/panda_shell
!git clone --depth 1 --filter=blob:none --branch dev/panda --sparse https://github.com/ferrandi/PandA-bambu.git
%cd PandA-bambu
!git sparse-checkout set documentation/bambu101
%cd ..
!mv PandA-bambu/documentation/bambu101/* .

# TrueFloat Custom Floating-point arithmetic cores
The TrueFloat library is a C-based custom floating-point arithmetic library embedded in the Bambu HLS tool.
The library offers the possibility to generate floating-point cores for aritrarily defined floating-point representations where the user may select the exponent and mantissa bitwidth, and the exponent bias. Additionally, the rounding mode can be selected between nearest even and truncate, the representation may or may not include exceptional values (i.e. Infinity, NaN), and it may support or not subnormal numbers.

Synthesized C/C++ description is automatically transformed by the HLS compiler starting from a standard float/double data type implementation through ad-hoc command line options that let the user specify the custom floating-point representation with function scope granularity.

The complete information on the available API for custom floating-point designs generation is available below:
```
--fp-exception-mode=<ieee|saturation|overflow>
    Set the soft-based exception handling mode:
            ieee    - IEEE754 standard exceptions (default)
        saturation - Inf is replaced with max value, Nan becomes undefined behaviour
        overflow  - Inf and Nan results in undefined behaviour

--fp-rounding-mode=<nearest_even|truncate>
    Set the soft-based rounding handling mode:
        nearest_even - IEEE754 standard rounding mode (default)
            truncate  - No rounding is applied

--fp-format=<func_name>*e<exp_bits>m<frac_bits>b<exp_bias><rnd_mode><exc_mode><?spec><?sign>
    Define arbitrary precision floating-point format by function (use comma separated
    list for multiple definitions). (i.e.: e8m27b-127nihs represent IEEE754 single precision FP)
        func_name - Set arbitrary floating-point format for a specific function (using
                    @ symbol here will resolve to the top function)
                    (Arbitrary floating-point format will apply to specified function
                    only, use --propagate-fp-format to extend it to called functions)
        exp_bits - Number of bits used by the exponent
        frac_bits - Number of bits used by the fractional value
        exp_bias - Bias applied to the unsigned value represented by the exponent bits
        rnd_mode - Rounding mode (exclusive option):
                        n - nearest_even: IEEE754 standard rounding mode
                        t - truncate    : no rounding is applied
        exc_mode - Exception mode (exclusive option):
                        i - ieee      : IEEE754 standard exceptions
                        a - saturation: Inf is replaced with max value, Nan becomes undefined behaviour
                        o - overflow  : Inf and Nan results in undefined behaviour
            spec   - Floating-point specialization string (multiple choice):
                        h - hidden one: IEEE754 standard representation with hidden one
                        s - subnormals: IEEE754 subnormal numbers
            sign   - Static sign representation (exclusive option):
                        - IEEE754 dynamic sign is used if omitted
                        1 - all values are considered as negative numbers
                        0 - all values are considered as positive numbers

--fp-format=inline-math
    The "inline-math" flag may be added to fp-format option to force floating-point
    arithmetic operators always inline policy

--fp-format=inline-conversion
    The "inline-conversion" flag may be added to fp-format option to force floating-point
    conversion operators always inline policy

--fp-format-interface
    User-defined floating-point format is applied to top interface signature if required
    (default modifies top function body only)

--fp-format-propagate
    Propagate user-defined floating-point format to called function when possible
```

## Sine Wave Generator Application
The sine wave generator application is a very simple toy example to show the capabilities of the TrueFloat library embedded into the Bambu HLS tool.
```
Usage: sine_gen <samples> <frequency> <amplitude> <sampling rate> <out_file>
```

The application generates a set of points sampling a sine wave with given frequency and amplitude to show the effects of different numerical representations on the accelerator performance and on the output results.

## Generating the baseline
To generate a basic version of the hardware design it is necessary to specify just a few options:
 - Input file: *sine_table_gen.cpp*
 - Top function name: *--top-fname=sine_table_gen*
 - Lib Math linking: _-lm_
 - Testbench program: *--generate-tb=sine_gen.cpp*
 - Testbench program arguments: *--tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=baseline.dat*

In [ ]:
%mkdir -p /content/truefloat/baseline
%cd /content/truefloat/baseline
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=baseline.dat --simulate --print-dot |& tee log.txt

Now we may run the simulation to get the baseline result

In [ ]:
%cd /content/truefloat
from multi_plot import main as plot_waves
plot_waves(["baseline/baseline.dat"])

## Generate TrueFloat custom floating-point variants
After the baseline has been generated you may generate some variants with a custom precision floating-point format of your choice passing the `--fp-format` command line option.

As an example, we can generate the following variants:
- e5m5b-15nih
- e4m4b-7nih
- e5m3b-15nih

Since the above custom formats are far from comparable in precision with the original double precision representation, it may be useful to disable the embedded automatic verification against the input representation, since it will be way off the golden values. To do so use the `-DCUSTOM_VERIFICATION` command line option.

In [ ]:
%mkdir -p /content/truefloat/e5m5b_15nih
%cd /content/truefloat/e5m5b_15nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e5m5b-15nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e5m5b_15nih.dat --simulate 

In [ ]:
%mkdir -p /content/truefloat/e4m4b_7nih
%cd /content/truefloat/e4m4b_7nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e4m4b-7nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e4m4b_7nih.dat --simulate --print-dot |& tee log.txt

In [ ]:
%mkdir -p /content/truefloat/e5m3b_15nih
%cd /content/truefloat/e5m3b_15nih
!bambu ../sine_table_gen.cpp --top-fname=sine_table_gen -lm --fp-format=@*e5m3b-15nih --fp-format-propagate -DCUSTOM_VERIFICATION --generate-tb=../sine_gen.cpp --tb-arg=50 --tb-arg=2 --tb-arg=2 --tb-arg=50 --tb-arg=e5m3b_15nih.dat --simulate --print-dot |& tee log.txt

## Look at the output
After the different variants have been generated, it is time to have a look at the generated output to verify the impact of the different custom floating-point formats on the application results.

In [ ]:
from pathlib import Path

base_dir = Path("/content/truefloat")
dat_files = []

for subdir in base_dir.iterdir():
    if subdir.is_dir():
        dat_files.extend(subdir.glob("*.dat"))

plot_waves(dat_files)